In [ ]:
%pip install jsonlines
%pip install language-tool-python

In [ ]:
import json
import jsonlines
from tqdm import tqdm
import spacy
import os
import pandas as pd
import re

nlp = spacy.load("en_core_web_sm")

input_path = "input_data/final_summaries.jsonl"
output_path = "output_data/corrected_summaries.jsonl"
comparison_csv = "output_data/summary_comparison.csv"

os.makedirs("output_data", exist_ok=True)

def fix_sentence_structures(text):
    text = re.sub(r'\b(\w+)\s+(dismissed?|filed|allowed?|given|executed?|awarded)\b', 
                  lambda m: f"{m.group(1)} was {m.group(2)}d" if not m.group(2).endswith('ed') else m.group(0), 
                  text, flags=re.IGNORECASE)
    
    patterns = [
        (r'\bcourt\s+dismiss\b', 'court was dismissed'),
        (r'\bcourt\s+allow\b', 'court was allowed'),
        (r'\bappeal\s+file\b', 'appeal was filed'),
        (r'\bappeal\s+dismiss\b', 'appeal was dismissed'),
        (r'\bcase\s+dismiss\b', 'case was dismissed'),
        (r'\bcase\s+file\b', 'case was filed'),
        (r'\bjudgment\s+pass\b', 'judgment was passed'),
        (r'\bdecree\s+grant\b', 'decree was granted'),
        (r'\bpetition\s+allow\b', 'petition was allowed'),
        (r'\bcompensation\s+award\b', 'compensation was awarded'),
    ]
    

def fix_agreement(text):
    doc = nlp(text)
    corrected = []
    
    for i, token in enumerate(doc):
        if token.pos_ == "VERB":
            if i > 0:
                prev_tokens = [t.text.lower() for t in doc[max(0, i-3):i]]
                singular_subjects = ['court', 'judge', 'appellant', 'respondent', 'defendant', 'case', 'appeal', 'judgment']
                has_singular = any(s in prev_tokens for s in singular_subjects)
                
                if has_singular:
                    if token.lemma_ == "be":
                        corrected.append("was" if any(t in prev_tokens for t in ['was', 'were']) else "is")
                    elif token.tag_ == "VBP":
                        corrected.append(token.lemma_)
                    else:
                        corrected.append(token.text)
                else:
                    corrected.append(token.text)
            else:
                corrected.append(token.text)
        else:
            corrected.append(token.text)
    
    return " ".join(corrected)

def cleanup_text(text):
    text = text.strip()
    if text:
        text = text[0].upper() + text[1:]
    
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    text = re.sub(r'([.,!?;:])\s*([A-Z])', r'\1 \2', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\b(\w+)(\s+\1)+\b', r'\1', text, flags=re.IGNORECASE)
    
    return text

def improve_flow(text):
    legal_terms = {
        'court': 'Court', 'judge': 'Judge', 'appeal': 'Appeal',
       
    }
    
    for word, proper in legal_terms.items():
        text = re.sub(rf'\b{word}\b', proper, text, flags=re.IGNORECASE)
    
    return text

def correct_summary(record):
    abstractive = record.get("abstractive_summary", "")
    extractive = record.get("extractive_summary", "")
    
    combined = abstractive if abstractive.strip() else extractive
    if not combined.strip():
        return ""
    
    text = fix_sentence_structures(combined)
    text = fix_agreement(text)
    text = improve_flow(text)
    text = cleanup_text(text)
    
    return text

results = []

with jsonlines.open(input_path) as reader:
    summaries = list(reader)

with jsonlines.open(output_path, mode='w') as writer:
    for record in tqdm(summaries, desc="Processing"):
        case_id = record.get("case_id", "N/A")
        original_abs = record.get("abstractive_summary", "")
        original_ext = record.get("extractive_summary", "")
        
        corrected = correct_summary(record)
        
        writer.write({
            "case_id": case_id,
            "original_abstractive": original_abs,
            "original_extractive": original_ext,
            "corrected_summary": corrected
        })
        
        results.append({
            "Case No.": case_id,
            "Original Abstractive Summary": original_abs,
            "Original Extractive Summary": original_ext,
            "Corrected Summary": corrected
        })

df = pd.DataFrame(results)
df.to_csv(comparison_csv, index=False)

In [ ]:
import jsonlines
import numpy as np
import pandas as pd
from numpy import dot
from numpy.linalg import norm
import os

input_file = "input_data/nlm_results_complete.jsonl"
all_output_file = "output_data/similarity_all.csv"
topk_output_file = "output_data/similarity_top10.csv"
TOP_DISPLAY = 3
TOP_SAVE = 10
query_cases = ["C.A._1022_2012", "C.A._1125_2014"]

os.makedirs("output_data", exist_ok=True)

def cosine_similarity(vec1, vec2):
    return dot(vec1, vec2) / (norm(vec1) * norm(vec2))

case_embeddings = {}

with jsonlines.open(input_file) as reader:
    for record in reader:
        case_id = record['case_id']
        sentences = record['sentence_embeddings']
        if sentences:
            doc_embedding = np.mean(np.array(sentences), axis=0)
            case_embeddings[case_id] = doc_embedding

all_results = []
topk_results = []

for query_case in query_cases:
    if query_case not in case_embeddings:
        print(f"Warning: Query case {query_case} not found!")
        continue

    query_vec = case_embeddings[query_case]

    similarities = [
        (other_id, cosine_similarity(query_vec, emb))
        for other_id, emb in case_embeddings.items() if other_id != query_case
    ]

    for sim_case, score in similarities:
        all_results.append({
            "Query Case No.": query_case,
            "Similar Case No.": sim_case,
            "Similarity Score": round(score, 4)
        })

    top_k = sorted(similarities, key=lambda x: x[1], reverse=True)[:TOP_SAVE]
    for sim_case, score in top_k:
        topk_results.append({
            "Query Case No.": query_case,
            "Similar Case No.": sim_case,
            "Similarity Score": round(score, 4)
        })

pd.DataFrame(all_results).to_csv(all_output_file, index=False)
pd.DataFrame(topk_results).to_csv(topk_output_file, index=False)

print("\n>> Top 3 similar cases per query:")
top3_display = pd.DataFrame(topk_results).groupby("Query Case No.").head(TOP_DISPLAY)
print(top3_display.to_string(index=False))

print(f"\n All similarity scores saved to: {all_output_file}")
print(f">> Top-10 similar cases saved to: {topk_output_file}")